# CrewAI Agentic RAG Ã¢â‚¬â€ SEC Filings

Role-based agentic RAG using CrewAI for SEC 10-K/10-Q question answering.

**Agent Architecture (Sequential Crew):**
1. **QueryPlanner** Ã¢â‚¬â€ Decomposes questions into sub-queries with ticker/year/form_type hints
2. **FinancialResearcher** Ã¢â‚¬â€ Executes hybrid BM25 + dense retrieval via `hybrid_search` tool
3. **FinancialAnalyst** Ã¢â‚¬â€ Generates precise answers from retrieved context
4. **QualityReviewer** Ã¢â‚¬â€ Critiques and repairs the answer in a single combined step
5. **EvaluationJudge** Ã¢â‚¬â€ Scores candidate answers against golden references (eval only)

**Framework contrast vs LangGraph/LlamaIndex:**
- LangGraph: explicit state machine with conditional routing
- LlamaIndex: event-driven workflow with typed events
- CrewAI: role-based agents with delegated tasks, sequential process


In [ ]:
pip install -q -U google-genai

In [ ]:
from google import genai
from google.genai import types as genai_types
print('google-genai imported successfully')

In [ ]:
# !pip uninstall -y chromadb chroma-hnswlib
# !pip install --no-cache-dir chromadb

In [ ]:
# Ã¢â€â‚¬Ã¢â€â‚¬ Install dependencies (run once) Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
!pip install crewai crewai-tools rank_bm25 sentence-transformers #chromadb
!pip install langchain-groq langchain-google-genai python-dotenv tqdm pandas
!pip install litellm   # required by crewai LLM class

In [ ]:
import os
import re
import json
import sys
import time
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Literal, Type

import chromadb
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from pydantic import BaseModel, Field, field_validator

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import BaseTool

warnings.filterwarnings('ignore')
load_dotenv()

# Resolve project root robustly when notebook runs from subfolder
# Look for unique project markers to find the correct root
def detect_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        has_config = (p / "config.py").exists()
        has_retriever = (p / "shared_retriever.py").exists()
        has_sec_data = (p / "sec_rag_team_share").exists()
        if has_config and has_retriever and has_sec_data:
            return p
    return start

PROJECT_ROOT = detect_project_root(Path.cwd())
print(f"PROJECT_ROOT detected as: {PROJECT_ROOT}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import CONFIG as SHARED_CONFIG
from shared_retriever import initialize_corpus

print("Libraries and shared modules loaded.")

In [ ]:
CONFIG: Dict[str, Any] = {
    'random_seed': 42,

    # Pilot vs full run
    'use_pilot':             True,
    'pilot_n_questions':     10,
    'full_n_questions':      80,
    'use_llm_judge':         True,
    'use_few_shot_examples': True,
    'pilot_judge_sample_n':  1,
    'full_judge_sample_n':   2,

    # 'dev' (faster/cheaper) or 'final' (best quality)
    'execution_profile': 'dev',
    'provider':          os.getenv('LLM_PROVIDER', 'gemini'),

    # Provider fallback order and per-provider RPM caps (matches langgraph_agentic_rag_sec_v3)
    'provider_fallback_order': ['gemini', 'groq'],
    'provider_rpm':            {'groq': 28, 'gemini': 10},

    # SEC dataset paths Ã¢â‚¬â€ update to match your environment
    'sec_chunks_path':   os.getenv('SEC_CHUNKS_PATH',   r'../sec_rag_team_share/sec_data/chunks/sec_chunks.jsonl'),
    'chroma_db_path':    os.getenv('CHROMA_DB_PATH',    r'../sec_rag_team_share/chroma_db'),
    'sec_eval_csv_path': os.getenv('SEC_EVAL_CSV_PATH', r'../sec_rag_team_share/evaluation/GenAI Eval QA.csv'),
    'results_dir':       os.getenv('RESULTS_DIR',       r'./results'),

    # Eval split config (matches LangGraph v3)
    'train_split':               'dev',
    'eval_split':                'test',
    'verbose_eval_logging':      True,
    'auto_export_results_input': True,

    # Retrieval knobs
    'bm25_top_k':                       int(os.getenv('BM25_TOP_K',                '8')),
    'dense_top_k':                      int(os.getenv('DENSE_TOP_K',               '8')),
    'rerank_top_k':                     int(os.getenv('RERANK_TOP_K',              '5')),
    'decomposition_top_k_per_subquery': int(os.getenv('DECOMP_TOP_K_PER_SUBQUERY', '4')),

    # Embedding / reranker (MUST match what was used to build ChromaDB Ã¢â‚¬â€ 768-dim)
    'dense_model_name':    'sentence-transformers/all-mpnet-base-v2',
    'reranker_model_name': 'cross-encoder/ms-marco-MiniLM-L-6-v2',

    # Groq model names (kept as fallback)
    'groq_dev_generator_model':   'llama-3.1-8b-instant',
    'groq_dev_agent_model':       'llama-3.1-8b-instant',
    'groq_dev_judge_model':       'llama-3.1-8b-instant',
    'groq_final_generator_model': 'meta-llama/llama-4-scout-17b-16e-instruct',
    'groq_final_agent_model':     'llama-3.1-8b-instant',
    'groq_final_judge_model':     'llama-3.1-8b-instant',
    'groq_fallback_agent_models':     ['llama-3.1-8b-instant'],
    'groq_fallback_generator_models': ['llama-3.1-8b-instant'],
    'groq_fallback_judge_models':     ['llama-3.1-8b-instant'],

    # Gemini model names Ã¢â‚¬â€ all use gemini-2.5-flash-lite
    'gemini_dev_generator_model':   'gemini-2.5-flash-lite',
    'gemini_dev_agent_model':       'gemini-2.5-flash-lite',
    'gemini_dev_judge_model':       'gemini-2.5-flash-lite',
    'gemini_final_generator_model': 'gemini-2.5-flash-lite',
    'gemini_final_agent_model':     'gemini-2.5-flash-lite',
    'gemini_final_judge_model':     'gemini-2.5-flash-lite',
    'gemini_fallback_agent_models':     [],
    'gemini_fallback_generator_models': [],
    'gemini_fallback_judge_models':     [],

    # Gemini pricing (USD per 1M tokens, as of 2025)
    'gemini_cost_input_per_1m':  0.10,   # gemini-2.5-flash-lite input
    'gemini_cost_output_per_1m': 0.40,   # gemini-2.5-flash-lite output

    # Temperatures Ã¢â‚¬â€ Gemini via LiteLLM returns empty candidates at exactly 0.0;
    # use 0.1 as the effective minimum for reliability.
    'temperature_planner':   0.1,
    'temperature_generator': 0.2,
    'temperature_critic':    0.1,
    'temperature_judge':     0.1,

    # Context window caps
    'generator_max_context_chunks': 8,
    'generator_max_context_chars':  12000,
    'control_max_context_chunks':   4,
    'control_max_context_chars':    6000,
    'judge_max_context_chunks':     3,
    'judge_max_context_chars':      4000,

    # Rate limiting / pacing
    'max_rpm':                  10,
    'inter_question_sleep_sec': 1.5,
    'llm_max_retries':          3,
    'llm_retry_base_delay_sec': 5,
}

CONFIG['judge_sample_n'] = (
    CONFIG['pilot_judge_sample_n'] if CONFIG['use_pilot'] else CONFIG['full_judge_sample_n']
) if CONFIG['use_llm_judge'] else 0

print(f'Config loaded. Provider: {CONFIG["provider"]}, Profile: {CONFIG["execution_profile"]}')

In [ ]:
# Ã¢â€â‚¬Ã¢â€â‚¬ Dataclass Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
@dataclass
class RetrievedChunk:
    doc_name:         str
    company:          str
    ticker:           str
    form_type:        str
    filing_year:      int
    page_num:         int
    chunk_id:         str
    raw_chunk:        str
    contextual_chunk: str
    score:            float
    source:           str


# Ã¢â€â‚¬Ã¢â€â‚¬ Pydantic Schemas Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
class SubQuery(BaseModel):
    query:       Optional[str] = Field(default=None)
    ticker:      Optional[str] = Field(default=None)
    filing_year: Optional[int] = Field(default=None)
    form_type:   Optional[str] = Field(default=None)

    @field_validator('ticker', mode='before')
    @classmethod
    def _norm_ticker(cls, v):
        if v is None: return None
        return str(v).strip().upper() or None

    @field_validator('form_type', mode='before')
    @classmethod
    def _norm_form(cls, v):
        if v is None: return None
        c = str(v).strip().upper()
        return c if c in {'10-K', '10-Q'} else None


class PlannerOutput(BaseModel):
    needs_decomposition: bool = False
    sub_queries: List[SubQuery] = Field(default_factory=list)


class CriticRepairOutput(BaseModel):
    decision:     Literal['accept', 'repair', 'insufficient_data'] = 'accept'
    reason:       str = ''
    final_answer: str = ''


class JudgeOutput(BaseModel):
    score:          float = 0.0
    claims_covered: float = 0.0
    reason:         str   = ''


# Ã¢â€â‚¬Ã¢â€â‚¬ Helpers Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
def safe_int(v, default: int = 0) -> int:
    try: return int(v)
    except: return default


def _rrf_score(rank: int, k: int = 60) -> float:
    return 1.0 / (k + rank + 1)


def format_retrieved_context(
    chunks: List[RetrievedChunk],
    max_chunks: int = 8,
    max_chars:  int = 12000,
) -> str:
    if not chunks:
        return 'No relevant context found.'
    parts, total = [], 0
    for i, c in enumerate(chunks[:max_chunks]):
        text = c.contextual_chunk or c.raw_chunk
        if total + len(text) > max_chars:
            rem = max_chars - total
            if rem > 100: text = text[:rem] + '...'
            else: break
        parts.append(f'[{i+1}] {text}')
        total += len(text)
    return '\n\n'.join(parts)


def extract_retrieved_doc_names(chunks: List[RetrievedChunk]) -> List[str]:
    seen: Dict[str, bool] = {}
    for c in chunks:
        if c.doc_name not in seen:
            seen[c.doc_name] = True
    return list(seen.keys())


def _try_parse_json(text: str, fallback: dict) -> dict:
    if not text: return fallback
    cleaned = re.sub(r'```(?:json)?\s*', '', text, flags=re.IGNORECASE).strip()
    cleaned = re.sub(r'```\s*$', '', cleaned).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    m = re.search(r'\{.*\}', cleaned, re.DOTALL)
    if m:
        try: return json.loads(m.group(0))
        except json.JSONDecodeError: pass
    return fallback


print('Schemas and helpers defined.')

In [ ]:
# CorpusIndex is now imported from shared_retriever
# (no duplicate needed here)
print('CorpusIndex imported from shared_retriever.')

In [ ]:
# Ã¢â€â‚¬Ã¢â€â‚¬ LLM Setup Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
provider = CONFIG['provider']
profile  = CONFIG['execution_profile']

if provider == 'groq':
    _gen_name   = CONFIG[f'groq_{profile}_generator_model']
    _agent_name = CONFIG[f'groq_{profile}_agent_model']
    _judge_name = CONFIG[f'groq_{profile}_judge_model']
    _key = os.getenv('GROQ_API_KEY', '')
    generator_llm = LLM(model=f'groq/{_gen_name}',   api_key=_key, temperature=CONFIG['temperature_generator'])
    agent_llm     = LLM(model=f'groq/{_agent_name}', api_key=_key, temperature=CONFIG['temperature_planner'])
    critic_llm    = LLM(model=f'groq/{_agent_name}', api_key=_key, temperature=CONFIG['temperature_critic'])
    judge_llm     = LLM(model=f'groq/{_judge_name}', api_key=_key, temperature=CONFIG['temperature_judge'])
    _genai_client = None  # not used for groq
else:
    _gen_name   = CONFIG[f'gemini_{profile}_generator_model']
    _agent_name = CONFIG[f'gemini_{profile}_agent_model']
    _judge_name = CONFIG[f'gemini_{profile}_judge_model']
    _key = os.getenv('GEMINI_API_KEY', '')
    generator_llm = LLM(model=f'gemini/{_gen_name}',   api_key=_key, temperature=CONFIG['temperature_generator'])
    agent_llm     = LLM(model=f'gemini/{_agent_name}', api_key=_key, temperature=CONFIG['temperature_planner'])
    critic_llm    = LLM(model=f'gemini/{_agent_name}', api_key=_key, temperature=CONFIG['temperature_critic'])
    judge_llm     = LLM(model=f'gemini/{_judge_name}', api_key=_key, temperature=CONFIG['temperature_judge'])
    # Direct genai.Client for judge calls (gives access to usage_metadata for token logging)
    _genai_client = genai.Client(api_key=_key)
    print(f'genai.Client initialised for judge calls (model: {_judge_name})')

print(f'LLMs: generator={_gen_name}, agent={_agent_name}, judge={_judge_name}')

# Ã¢â€â‚¬Ã¢â€â‚¬ Embedding + Reranker (module-level, used directly by CorpusIndex) Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
print('Loading embedding model...')
dense_model = SentenceTransformer(CONFIG['dense_model_name'])
print('Loading reranker...')
reranker = CrossEncoder(CONFIG['reranker_model_name'])
print('Models ready.')

In [ ]:
# Ã¢â€â‚¬Ã¢â€â‚¬ Additional imports Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
import threading
import litellm
from collections import deque
from collections import Counter as _Counter
import re as _re


# Ã¢â€â‚¬Ã¢â€â‚¬ Rate Limiter (matches langgraph_agentic_rag_sec_v3) Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
class RateLimiter:
    """Sliding-window RPM throttle. Thread-safe."""
    def __init__(self, max_rpm: int):
        self.max_rpm = max_rpm
        self.window  = 60.0
        self._ts: deque = deque()
        self._lock = threading.Lock()

    def wait(self):
        with self._lock:
            now = time.time()
            while self._ts and now - self._ts[0] >= self.window:
                self._ts.popleft()
            if len(self._ts) >= self.max_rpm:
                sleep_for = self.window - (now - self._ts[0])
                if sleep_for > 0:
                    print(f'  [RateLimit] {self.max_rpm} RPM cap Ã¢â‚¬â€ waiting {sleep_for:.1f}s')
                    time.sleep(sleep_for)
            self._ts.append(time.time())


_rate_limiters: Dict[str, RateLimiter] = {
    p: RateLimiter(max_rpm=rpm)
    for p, rpm in CONFIG['provider_rpm'].items()
}

def get_rate_limiter(provider: str) -> RateLimiter:
    return _rate_limiters.get(provider, RateLimiter(max_rpm=CONFIG['max_rpm']))


# Ã¢â€â‚¬Ã¢â€â‚¬ Error Detection Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
def is_rate_limit_error(exc: Exception) -> bool:
    msg = str(exc).lower()
    return 'rate limit' in msg or 'rate_limit' in msg or '429' in msg

def is_model_exhausted_error(exc: Exception) -> bool:
    msg = str(exc).lower()
    return any(m in msg for m in [
        'tokens per day', 'tpd', 'quota exceeded',
        'resource_exhausted', 'generatecontentinputtokens',
    ])


# Ã¢â€â‚¬Ã¢â€â‚¬ Per-Question Token Accumulator Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
# Each step (crew, judge_correctness, judge_faithfulness) accumulates tokens here.
# Reset at the start of each question; read at the end to build cost columns.

_question_tokens: Dict[str, Dict[str, int]] = {}

def _reset_question_tokens() -> None:
    global _question_tokens
    _question_tokens = {}

def _add_tokens(step: str, input_tok: int, output_tok: int) -> None:
    if step not in _question_tokens:
        _question_tokens[step] = {'input': 0, 'output': 0}
    _question_tokens[step]['input']  += int(input_tok  or 0)
    _question_tokens[step]['output'] += int(output_tok or 0)

def _get_question_token_totals() -> tuple:
    """Return (total_input, total_output) summed across all steps for current question."""
    total_in  = sum(v['input']  for v in _question_tokens.values())
    total_out = sum(v['output'] for v in _question_tokens.values())
    return total_in, total_out

def _estimate_cost_usd(total_input: int, total_output: int) -> float:
    rate_in  = CONFIG.get('gemini_cost_input_per_1m',  0.10) / 1_000_000
    rate_out = CONFIG.get('gemini_cost_output_per_1m', 0.40) / 1_000_000
    return total_input * rate_in + total_output * rate_out


# Ã¢â€â‚¬Ã¢â€â‚¬ LiteLLM Success Callback Ã¢â‚¬â€ captures crew agent token usage Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
def _crewai_token_callback(kwargs, completion_response, start_time, end_time):
    """Intercepts every LiteLLM call made by CrewAI agents and accumulates tokens."""
    try:
        usage = getattr(completion_response, 'usage', None) or {}
        if isinstance(usage, dict):
            in_tok  = usage.get('prompt_tokens', 0) or 0
            out_tok = usage.get('completion_tokens', 0) or 0
        else:
            in_tok  = getattr(usage, 'prompt_tokens',     0) or 0
            out_tok = getattr(usage, 'completion_tokens', 0) or 0
        if in_tok or out_tok:
            _add_tokens('crew', in_tok, out_tok)
    except Exception:
        pass

litellm.success_callback = [_crewai_token_callback]


# Ã¢â€â‚¬Ã¢â€â‚¬ Model Preference + Disable Tracking (matches langgraph_agentic_rag_sec_v3) Ã¢â€â‚¬
_preferred_models_by_task: Dict[str, str] = {}
_disabled_model_keys: set = set()

def disable_model_for_session(provider: str, model_name: str) -> None:
    key = f'{provider}::{model_name}'
    _disabled_model_keys.add(key)
    print(f'  [Model] Disabled {key} for this session.')

def get_task_model_snapshot(task_names: List[str]) -> Dict[str, str]:
    return {t: _preferred_models_by_task.get(t, 'not_set') for t in task_names}


# Ã¢â€â‚¬Ã¢â€â‚¬ Crew LLM Candidates (agent + generator pairs, for kickoff() fallback) Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
def get_crew_model_candidates() -> List[tuple]:
    candidates = []
    seen: set = set()
    profile = CONFIG['execution_profile']
    for prov in CONFIG.get('provider_fallback_order', [CONFIG['provider']]):
        primary_agent = CONFIG.get(f'{prov}_{profile}_agent_model', '')
        primary_gen   = CONFIG.get(f'{prov}_{profile}_generator_model', '')
        fb_agents = [primary_agent] + CONFIG.get(f'{prov}_fallback_agent_models', [])
        fb_gens   = [primary_gen]   + CONFIG.get(f'{prov}_fallback_generator_models', [])
        for a_model, g_model in zip(fb_agents, fb_gens):
            key = f'{prov}::{a_model}::{g_model}'
            if key not in seen and a_model and g_model:
                candidates.append((a_model, g_model, prov))
                seen.add(key)
    return candidates


# Ã¢â€â‚¬Ã¢â€â‚¬ Judge Schema Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
class JudgeOutput(BaseModel):
    score:          float = Field(default=0.0, description='Score 0.0-1.0')
    claims_covered: float = Field(default=0.0, description='Fraction of key facts covered 0.0-1.0')
    reason:         str   = Field(default='',  description='Short explanation')


# Ã¢â€â‚¬Ã¢â€â‚¬ Judge Prompt Builders Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
def _build_correctness_prompt(question: str, reference_answer: str, candidate_answer: str) -> str:
    return (
        'Score the candidate answer against the reference answer from 0 to 1 for a financial QA task. '
        '1 = correct value, correct units, correct period. '
        '0 = wrong number, wrong company, wrong period, or completely missing. '
        'Give partial credit for answers close but with unit errors. '
        'Also set claims_covered: fraction of distinct facts/numbers in the reference present in the candidate. '
        'Return a valid JSON object only that matches the requested schema.\n\n'
        f'Question:\n{question}\n\n'
        f'Reference answer:\n{reference_answer}\n\n'
        f'Candidate answer:\n{candidate_answer}'
    )

def _build_faithfulness_prompt(question: str, context: str, candidate_answer: str) -> str:
    return (
        'Score how well the candidate answer is grounded in the retrieved filing context from 0 to 1. '
        '1 = every number and claim is directly supported by the context. '
        '0 = answer contains numbers or claims not present in the context (hallucinated). '
        'Also set claims_covered: fraction of claims in the candidate supported by the context. '
        'Return a valid JSON object only that matches the requested schema.\n\n'
        f'Question:\n{question}\n\n'
        f'Retrieved context:\n{context}\n\n'
        f'Candidate answer:\n{candidate_answer}'
    )


# Ã¢â€â‚¬Ã¢â€â‚¬ Safe Gemini Judge Call (rate limiting + retry + token logging) Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
def _safe_judge_call(prompt_text: str, task_name: str) -> JudgeOutput:
    """
    Call Gemini directly via genai.Client() for structured judge output.
    - Rate-limits via per-provider RateLimiter
    - Logs input/output tokens via _add_tokens()
    - Falls back to JSON parse if response.parsed is None
    """
    _fb = JudgeOutput(score=0.0, claims_covered=0.0, reason='judge fallback Ã¢â‚¬â€ all attempts failed')

    if _genai_client is None:
        # Groq fallback: no structured judge available without LangChain
        _preferred_models_by_task[task_name] = 'groq::skipped'
        return _fb

    profile     = CONFIG['execution_profile']
    judge_model = CONFIG.get(f'gemini_{profile}_judge_model', 'gemini-2.5-flash-lite')
    max_retries = CONFIG['llm_max_retries']

    for attempt in range(max_retries):
        try:
            get_rate_limiter('gemini').wait()
            response = _genai_client.models.generate_content(
                model=judge_model,
                contents=prompt_text,
                config=genai_types.GenerateContentConfig(
                    response_mime_type='application/json',
                    response_schema=JudgeOutput,
                    temperature=CONFIG.get('temperature_judge', 0.0),
                ),
            )

            print(f'JUDGE RESPONSE: {response}')

            # Ã¢â€â‚¬Ã¢â€â‚¬ Token logging Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
            if response.usage_metadata:
                in_tok  = response.usage_metadata.prompt_token_count        or 0
                out_tok = response.usage_metadata.candidates_token_count    or 0
                _add_tokens(task_name, in_tok, out_tok)
                if CONFIG.get('verbose_eval_logging'):
                    print(f'    [tokens/{task_name}] in={in_tok}  out={out_tok}')

            # Ã¢â€â‚¬Ã¢â€â‚¬ Parse result Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
            result = response.parsed
            if result is None:
                result_dict = json.loads(response.text)
                result = JudgeOutput(**result_dict)

            _preferred_models_by_task[task_name] = f'gemini::{judge_model}'
            return result

        except Exception as e:
            print(f'  [WARN] Judge ({task_name}) attempt {attempt+1}/{max_retries} '
                  f'on gemini:{judge_model} failed: {e}')
            if is_model_exhausted_error(e):
                break
            if attempt < max_retries - 1:
                delay = CONFIG['llm_retry_base_delay_sec'] * (2 ** attempt)
                time.sleep(delay)

    return _fb


# Ã¢â€â‚¬Ã¢â€â‚¬ Public Judge Functions Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
def llm_judge_correctness(
    question: str, reference_answer: str, candidate_answer: str,
) -> tuple:
    """
    Score candidate answer vs reference (correctness).
    Returns: (score, claims_covered, reason, model_used)
    """
    result = _safe_judge_call(
        _build_correctness_prompt(question, reference_answer, candidate_answer),
        task_name='judge_correctness',
    )
    model_used = _preferred_models_by_task.get('judge_correctness', 'unknown')
    return (
        max(0.0, min(1.0, float(result.score))),
        max(0.0, min(1.0, float(result.claims_covered))),
        result.reason,
        model_used,
    )


def llm_judge_faithfulness(
    question: str, context: str, candidate_answer: str,
) -> tuple:
    """
    Score faithfulness to retrieved context.
    Returns: (score, claims_covered, reason, model_used)
    """
    result = _safe_judge_call(
        _build_faithfulness_prompt(
            question,
            context[:CONFIG['judge_max_context_chars']],
            candidate_answer,
        ),
        task_name='judge_faithfulness',
    )
    model_used = _preferred_models_by_task.get('judge_faithfulness', 'unknown')
    return (
        max(0.0, min(1.0, float(result.score))),
        max(0.0, min(1.0, float(result.claims_covered))),
        result.reason,
        model_used,
    )


# Ã¢â€â‚¬Ã¢â€â‚¬ Scoring Utilities (from langgraph_agentic_rag_sec_v3) Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
def _tokenize(text: str) -> List[str]:
    return _re.sub(r'[^\w\s]', '', text.lower()).split()

def token_f1_score(pred: str, ref: str) -> float:
    if not pred or not ref:
        return 0.0
    pred_toks = _tokenize(pred)
    ref_toks  = _tokenize(ref)
    if not pred_toks or not ref_toks:
        return 0.0
    common = sum((_Counter(pred_toks) & _Counter(ref_toks)).values())
    if common == 0:
        return 0.0
    precision = common / len(pred_toks)
    recall    = common / len(ref_toks)
    return 2 * precision * recall / (precision + recall)

def numerical_accuracy_score(pred: str, ref: str) -> Optional[float]:
    nums_ref  = [float(n.replace(',', '')) for n in _re.findall(r'-?\d[\d,]*\.?\d*', ref)]
    nums_pred = [float(n.replace(',', '')) for n in _re.findall(r'-?\d[\d,]*\.?\d*', pred)]
    if not nums_ref:
        return None
    hits = sum(
        1 for r in nums_ref
        if any(abs(p - r) / (abs(r) + 1e-9) < 0.01 for p in nums_pred)
    )
    return hits / len(nums_ref)

def compute_decision_from_text(answer_text: str) -> str:
    lower = answer_text.lower()
    refusal_phrases = [
        'insufficient', 'not contain', 'not available', 'cannot find',
        "don't have", 'no information', 'not provided', 'unable to',
        'not enough', 'not present', 'not found', 'insufficient data',
    ]
    return 'refuse' if any(p in lower for p in refusal_phrases) else 'answer'


print('Rate limiter, token accumulator, genai judge, and scoring utilities ready.')

In [ ]:
# !pip uninstall -y chromadb chroma-hnswlib
# !pip install --no-cache-dir chromadb

In [ ]:
# â”€â”€ SEC Corpus Loading â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

print("Initializing CorpusIndex from shared_retriever...")
print("(shared_retriever handles path resolution automatically)")

# Call initialize_corpus with no explicit paths - it resolves them correctly
global_index = initialize_corpus()
print(f"CorpusIndex ready: {len(global_index.df):,} chunks with hybrid retrieval")

In [ ]:
# !pip uninstall -y chromadb chroma-hnswlib
# !pip install --no-cache-dir chromadb

In [ ]:
# print('hello')

In [ ]:
# import chromadb
# print("chromadb version:", chromadb.__version__)

# client = chromadb.PersistentClient(path="../sec_rag_team_share/chroma_db")
# print("client created")

# cols = client.list_collections()
# print("collections:", cols)

In [ ]:
# import chromadb
# from pathlib import Path

# test_path = Path("../sec_rag_team_share/chroma_db_fresh_test")
# test_path.mkdir(parents=True, exist_ok=True)

# client = chromadb.PersistentClient(path=str(test_path))
# print("fresh client OK")
# print(client.list_collections())

In [ ]:
# Ã¢â€â‚¬Ã¢â€â‚¬ Evaluation Data Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
# Mirrors LangGraph v3 evaluation setup.
# Falls back to hardcoded test questions if the CSV is not available.

FALLBACK_QUESTIONS = [
    {'id': 'FB_001', 'question': "What was NVIDIA's total revenue for fiscal year 2024?",
     'reference_answer': "NVIDIA's total revenue for FY2024 was $60.9 billion.",
     'question_type': 'extractive', 'company': 'NVIDIA', 'ticker': 'NVDA',
     'form_type': '10-K', 'filing_year': 2024},
    {'id': 'FB_002', 'question': "What were Apple's total net sales in fiscal year 2024?",
     'reference_answer': "Apple's total net sales in FY2024 were $391.0 billion.",
     'question_type': 'extractive', 'company': 'Apple', 'ticker': 'AAPL',
     'form_type': '10-K', 'filing_year': 2024},
    {'id': 'FB_003', 'question': "What was Microsoft's cloud revenue in FY2024?",
     'reference_answer': "Microsoft's Intelligent Cloud segment revenue was $105.4 billion in FY2024.",
     'question_type': 'extractive', 'company': 'Microsoft', 'ticker': 'MSFT',
     'form_type': '10-K', 'filing_year': 2024},
    {'id': 'FB_004', 'question': "What are NVIDIA's main risk factors as of their latest 10-K?",
     'reference_answer': "NVIDIA's risk factors include supply chain constraints, export controls, competition, and rapid technological change.",
     'question_type': 'qualitative', 'company': 'NVIDIA', 'ticker': 'NVDA',
     'form_type': '10-K', 'filing_year': 2024},
    {'id': 'FB_005', 'question': "Compare JPMorgan Chase and Bank of America's total assets in 2024.",
     'reference_answer': "JPMorgan Chase had total assets of approximately $3.9 trillion and Bank of America had approximately $3.3 trillion as of end of 2024.",
     'question_type': 'comparative', 'company': 'JPM/BAC', 'ticker': None,
     'form_type': '10-K', 'filing_year': 2024},
]

eval_df: Optional[pd.DataFrame] = None

try:
    raw_eval_df = pd.read_csv(CONFIG['sec_eval_csv_path'])
    raw_eval_df = raw_eval_df[raw_eval_df['question_id'].notna()].copy()
    raw_eval_df['question_id'] = raw_eval_df['question_id'].astype(str).str.strip()
    raw_eval_df = raw_eval_df[raw_eval_df['question_id'] != ''].copy()

    full_eval_df = raw_eval_df.rename(columns={
        'question_id':      'id',
        'expected_answer':  'reference_answer',
    }).copy()

    full_eval_df['id'] = full_eval_df['id'].apply(
        lambda x: f"SEC_{int(float(x)):03d}" if str(x).replace('.', '', 1).isdigit() else str(x)
    )
    for col in ('expected_decision', 'question_type', 'company', 'reference_answer', 'split'):
        if col in full_eval_df.columns:
            full_eval_df[col] = full_eval_df[col].fillna('').astype(str).str.strip()

    eval_source_df = full_eval_df[
        full_eval_df.get('split', pd.Series('', index=full_eval_df.index)) == CONFIG['eval_split']
    ].reset_index(drop=True)
    if eval_source_df.empty:
        eval_source_df = full_eval_df.reset_index(drop=True)

    n_sample = CONFIG['pilot_n_questions'] if CONFIG['use_pilot'] else CONFIG['full_n_questions']
    eval_df = eval_source_df.sample(
        n=min(n_sample, len(eval_source_df)),
        random_state=CONFIG['random_seed'],
    ).sort_values('id').reset_index(drop=True)

    print(f'Loaded eval CSV: {len(eval_df)} questions (split="{CONFIG["eval_split"]}")')
    print(eval_df[['id', 'question', 'reference_answer']].head(3).to_string(index=False))

except Exception as e:
    print(f'[Warn] Could not load eval CSV ({e}). Using fallback questions.')
    eval_df = pd.DataFrame(FALLBACK_QUESTIONS)
    print(f'Fallback eval set: {len(eval_df)} questions.')

In [ ]:
# Ã¢â€â‚¬Ã¢â€â‚¬ Global Run State Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
_global_state: Dict[str, Any] = {
    'index':            global_index,
    'retrieved_chunks': [],
    'question':         '',
}


# Ã¢â€â‚¬Ã¢â€â‚¬ CrewAI Tool: Hybrid Search Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
class HybridSearchInput(BaseModel):
    query:       str           = Field(description='Search query optimized for SEC filing retrieval')
    ticker:      Optional[str] = Field(default=None, description='Company ticker e.g. NVDA, AAPL. Use when company-specific.')
    filing_year: Optional[int] = Field(default=None, description='Calendar year the filing was submitted e.g. 2023, 2024.')
    form_type:   Optional[str] = Field(default=None, description='10-K for annual, 10-Q for quarterly.')


class HybridSearchTool(BaseTool):
    name:        str             = 'hybrid_search'
    description: str             = (
        'Search SEC 10-K and 10-Q filings using hybrid BM25 + dense retrieval with '
        'cross-encoder reranking. Call once per sub-query from the planner. '
        'Provide ticker, filing_year, form_type when known to focus retrieval. '
        'Returns the most relevant filing passages found so far (all accumulated calls).'
    )
    args_schema: Type[BaseModel] = HybridSearchInput

    def _run(
        self,
        query:       str,
        ticker:      Optional[str] = None,
        filing_year: Optional[int] = None,
        form_type:   Optional[str] = None,
    ) -> str:
        index: CorpusIndex = _global_state.get('index')
        if index is None:
            return 'Error: No corpus index available.'

        chunks = index.hybrid_search(
            query=query,
            bm25_top_k=CONFIG['bm25_top_k'],
            dense_top_k=CONFIG['dense_top_k'],
            rerank_top_k=CONFIG['rerank_top_k'],
            ticker=ticker,
            filing_year=filing_year,
            form_type=form_type,
        )

        # Accumulate chunks (dedup by chunk_id)
        existing = _global_state.get('retrieved_chunks', [])
        seen_ids = {c.chunk_id for c in existing}
        for c in chunks:
            if c.chunk_id not in seen_ids:
                existing.append(c)
                seen_ids.add(c.chunk_id)
        _global_state['retrieved_chunks'] = existing

        print(f'  [HybridSearch] query={query!r:.70}  ticker={ticker}  year={filing_year}  form={form_type}')
        print(f'  [HybridSearch] {len(chunks)} new chunks | running total: {len(existing)}')

        # Return ALL accumulated chunks so the analyst always sees the full
        # context across multiple sub-query searches, not just the latest call.
        return format_retrieved_context(
            existing,
            CONFIG['generator_max_context_chunks'],
            CONFIG['generator_max_context_chars'],
        )


search_tool = HybridSearchTool()
print('CrewAI tool defined.')

In [ ]:
# Ã¢â€â‚¬Ã¢â€â‚¬ CrewAI Agents Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
# Agents are built via _build_crew_agents() so build_and_run_crew() can
# recreate them with fallback LLMs without duplicating role/goal/backstory strings.

_COMPANIES = (
    'AAPL (Apple), BAC (Bank of America), BRK-B (Berkshire Hathaway), COST (Costco), '
    'CVX (Chevron), JNJ (Johnson & Johnson), JPM (JPMorgan Chase), MSFT (Microsoft), '
    'NVDA (NVIDIA), UNH (UnitedHealth), WMT (Walmart), XOM (ExxonMobil)'
)


def _build_crew_agents(agent_llm_obj, generator_llm_obj, critic_llm_obj):
    """
    Create fresh CrewAI Agent instances with the given LLM objects.
    Used for both primary setup and fallback retries.
    Returns: (planner, researcher, analyst, critic)
    """
    _verbose = CONFIG.get('verbose_eval_logging', True)

    planner = Agent(
        role='Query Decomposition Specialist',
        goal=(
            'Decompose complex financial questions into targeted sub-queries for SEC filing retrieval. '
            'Identify the companies, fiscal years, and filing types needed.'
        ),
        backstory=(
            f'You are a financial research planner specializing in SEC filings from: {_COMPANIES}. '
            'You analyze questions and determine whether they need data from multiple filings, '
            'then produce precise sub-queries with metadata hints (ticker, filing_year, form_type).'
        ),
        llm=agent_llm_obj,
        verbose=_verbose,
        allow_delegation=False,
        max_iter=3,
        respect_context_window=True,
    )

    researcher = Agent(
        role='SEC Filing Research Specialist',
        goal=(
            'Retrieve the most relevant SEC filing passages to answer financial questions. '
            'Use the hybrid_search tool once per sub-query identified by the planner.'
        ),
        backstory=(
            'You are a financial researcher with deep expertise in SEC 10-K and 10-Q filings. '
            'You use hybrid BM25 + dense retrieval with cross-encoder reranking. '
            'When the planner identifies multiple sub-queries (comparing companies or years), '
            'you call hybrid_search separately for each, passing the metadata filters.'
        ),
        llm=agent_llm_obj,
        tools=[search_tool],
        verbose=_verbose,
        allow_delegation=False,
        max_iter=6,
        respect_context_window=True,
    )

    analyst = Agent(
        role='Financial Analyst',
        goal=(
            'Generate precise, well-grounded answers from retrieved SEC filing context. '
            'Be exact with numbers, units (millions/billions), and fiscal periods.'
        ),
        backstory=(
            'You are a senior financial analyst who synthesizes data from SEC filings into clear answers. '
            'You provide precise figures with correct units and fiscal periods. '
            'You only answer from the provided context and explicitly state when data is missing. '
            'You never estimate or hallucinate financial figures.'
        ),
        llm=generator_llm_obj,
        verbose=_verbose,
        allow_delegation=False,
        max_iter=3,
        respect_context_window=True,
    )

    critic = Agent(
        role='Financial QA Quality Reviewer',
        goal=(
            'Evaluate financial answers for accuracy and provide corrections when needed. '
            'Choose: accept (correct), repair (fix errors), or insufficient_data (data absent).'
        ),
        backstory=(
            'You are a meticulous financial QA reviewer. You verify answers against retrieved context, '
            'checking for correct numbers, units, fiscal periods, and line-item names. '
            'When you find errors, you immediately provide a corrected version. '
            'Use insufficient_data only when the required financial data is genuinely absent.'
        ),
        llm=critic_llm_obj,
        verbose=_verbose,
        allow_delegation=False,
        max_iter=3,
        respect_context_window=True,
    )

    return planner, researcher, analyst, critic


# Primary instances (used on first attempt Ã¢â‚¬â€ no extra LLM object creation)
planner_agent, researcher_agent, analyst_agent, critic_agent = _build_crew_agents(
    agent_llm, generator_llm, critic_llm
)

# judge_agent kept for backward compatibility; judge is now done outside the crew
# via llm_judge_correctness / llm_judge_faithfulness (see rate-limiter cell above)
judge_agent = Agent(
    role='Financial QA Evaluation Judge',
    goal='Score candidate answers against reference answers on a 0-1 scale.',
    backstory=(
        'You are a strict financial QA judge who evaluates factual correctness: '
        'correct numbers, units, fiscal period, and company. '
        'Score 1.0 = fully correct, 0.5 = partial, 0.0 = wrong or no answer.'
    ),
    llm=judge_llm,
    verbose=False,
    allow_delegation=False,
    max_iter=2,
    respect_context_window=True,
)

print('CrewAI agents defined.')

In [ ]:
# Ã¢â€â‚¬Ã¢â€â‚¬ JSON Helper Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
def _try_parse_json(raw: str, default: Any) -> Any:
    """Attempt to parse JSON from a raw string; return default on failure."""
    if not raw:
        return default
    text = raw.strip()
    for fence in ('```json', '```'):
        if text.startswith(fence):
            text = text[len(fence):]
    text = text.rstrip('`').strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except Exception:
            pass
    return default


# Ã¢â€â‚¬Ã¢â€â‚¬ Result Parser Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
def _parse_crew_result(crew_result: Any, question: str) -> Dict[str, Any]:
    """Parse CrewAI crew output into a structured result dict (no in-crew judge)."""
    chunks = _global_state.get('retrieved_chunks', [])
    out: Dict[str, Any] = {
        'question':            question,
        'final_answer':        '',
        'generated_answer':    '',
        'critic_decision':     'accept',
        'critic_reason':       '',
        'repair_used':         False,
        'retrieved_chunks':    chunks,
        'retrieved_doc_names': extract_retrieved_doc_names(chunks),
        'needs_decomposition': False,
        'sub_queries':         [],
    }
    try:
        to = crew_result.tasks_output

        # Task 0 Ã¢â‚¬â€ Planner
        if len(to) > 0:
            pj = _try_parse_json(to[0].raw, {'needs_decomposition': False, 'sub_queries': []})
            out['needs_decomposition'] = bool(pj.get('needs_decomposition', False))
            out['sub_queries']         = pj.get('sub_queries', [])

        # Task 1 Ã¢â‚¬â€ Researcher (chunks captured via tool side-effect)

        # Task 2 Ã¢â‚¬â€ Analyst
        if len(to) > 2:
            out['generated_answer'] = to[2].raw.strip()
            out['final_answer']     = out['generated_answer']

        # Task 3 Ã¢â‚¬â€ Critic + Repair
        if len(to) > 3:
            cj = _try_parse_json(
                to[3].raw,
                {'decision': 'accept', 'reason': '', 'final_answer': out['generated_answer']},
            )
            out['critic_decision'] = cj.get('decision', 'accept')
            out['critic_reason']   = cj.get('reason', '')
            fa = cj.get('final_answer') or cj.get('revised_answer') or out['generated_answer']
            out['final_answer']    = str(fa).strip()
            out['repair_used']     = out['critic_decision'] == 'repair'

    except Exception as e:
        print(f'  [ParseResult] Warning: {e}')
    return out


# Ã¢â€â‚¬Ã¢â€â‚¬ CrewAI LLM factory Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
def _make_crewai_llm(model_name: str, provider: str, temperature: float) -> 'LLM':
    api_key = os.getenv('GROQ_API_KEY' if provider == 'groq' else 'GEMINI_API_KEY', '')
    return LLM(model=f'{provider}/{model_name}', api_key=api_key, temperature=temperature)


# Ã¢â€â‚¬Ã¢â€â‚¬ Crew Builder with Rate Limiting + Model Fallback Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
def build_and_run_crew(
    question: str,
    golden_answer: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Build and run a sequential CrewAI crew for one question.

    Key behaviours:
    - Always creates FRESH LLM objects and agent instances per question to prevent
      accumulated conversation history from causing empty LLM responses on later questions.
    - Resets per-question token accumulator before kickoff.
    - LiteLLM callback (_crewai_token_callback) accumulates crew tokens automatically.
    - Rate-limits kickoff() via per-provider RateLimiter.
    - Falls back to alternative models on rate-limit errors.
    - Judge scoring is done separately in the eval loop.
    """
    _reset_question_tokens()  # start fresh token count for this question
    _global_state['retrieved_chunks'] = []
    _global_state['question']         = question

    start_time = time.time()

    def _make_tasks(p_agent, r_agent, a_agent, c_agent) -> List:
        plan_task = Task(
            description=(
                f'Available companies: {_COMPANIES}.\n\n'
                f'Question: {question}\n\n'
                'Decide if this needs data from MULTIPLE distinct filings '
                '(different companies OR different fiscal years). Produce sub-queries.\n\n'
                'Rules:\n'
                '- Decompose only for 2+ companies or 2+ fiscal years.\n'
                '- Single-company/period: needs_decomposition=false, one sub-query.\n'
                '- filing_year = calendar year of submission '
                '(Apple FY2024 10-K filed Nov 2024 -> filing_year=2024).\n\n'
                'Return ONLY valid JSON with keys: needs_decomposition (bool), '
                'sub_queries (list of: query, ticker, filing_year, form_type).'
            ),
            expected_output='Valid JSON: {"needs_decomposition": bool, "sub_queries": [...]}',
            agent=p_agent,
        )
        research_task = Task(
            description=(
                f'Research question: {question}\n\n'
                'Review the planner output. Call hybrid_search once per sub-query. '
                'Pass ticker, filing_year, form_type as filters when provided. '
                'If needs_decomposition=true, call once per sub-query separately.'
            ),
            expected_output='Comprehensive retrieval context with relevant SEC filing passages.',
            context=[plan_task],
            agent=r_agent,
        )
        generate_task = Task(
            description=(
                f'Question: {question}\n\n'
                'Using the retrieved context, provide a precise answer.\n'
                '- Quote exact figures (do not round unless source rounds).\n'
                '- Specify units (millions, billions, %).\n'
                '- Specify the fiscal period (FY2024, Q2 2023, etc.).\n'
                '- Name the exact line item from the filing.\n'
                '- If data is absent: "The retrieved context does not contain this information."'
            ),
            expected_output='Precise, well-grounded answer with exact figures, units, and periods.',
            context=[research_task],
            agent=a_agent,
        )
        critic_task = Task(
            description=(
                f'Question: {question}\n\n'
                'Evaluate the generated answer using the retrieved context. Check:\n'
                '1. Correct numbers and units (millions vs billions, sign).\n'
                '2. Correct fiscal period (FY2023 vs FY2024, Q1 vs Q2).\n'
                '3. Correct company and line-item names.\n'
                '4. Whether the answer is grounded in context.\n\n'
                'Choose ONE decision:\n'
                '- accept: factually correct and well-grounded.\n'
                '- repair: data present but answer has errors -- provide corrected answer.\n'
                '- insufficient_data: required data genuinely absent from context.\n\n'
                'Return ONLY valid JSON: decision (string), reason (string), final_answer (string).\n'
                'final_answer: copy original if accept, corrected if repair, '
                '"Insufficient data: [reason]" if insufficient_data.'
            ),
            expected_output='Valid JSON: {"decision": str, "reason": str, "final_answer": str}',
            context=[research_task, generate_task],
            agent=c_agent,
        )
        return [plan_task, research_task, generate_task, critic_task]

    candidates  = get_crew_model_candidates()
    last_error: Optional[Exception] = None

    for attempt_idx, (agent_model, gen_model, provider) in enumerate(candidates):
        if attempt_idx > 0:
            print(f'  [CrewFallback] Attempt {attempt_idx + 1}: '
                  f'agent={provider}:{agent_model}  gen={provider}:{gen_model}')
            _global_state['retrieved_chunks'] = []

        # Always build fresh LLM objects and agent instances per question.
        # Reusing agents across crew.kickoff() calls accumulates conversation
        # history internally, causing Gemini to silently return None on later questions.
        try:
            _agent_llm = _make_crewai_llm(agent_model, provider, CONFIG['temperature_planner'])
            _gen_llm   = _make_crewai_llm(gen_model,   provider, CONFIG['temperature_generator'])
            _crit_llm  = _make_crewai_llm(agent_model, provider, CONFIG['temperature_critic'])
        except Exception as e:
            print(f'  [CrewFallback] Could not build LLMs for {provider}:{agent_model}: {e}')
            last_error = e
            continue

        p_a, r_a, a_a, c_a = _build_crew_agents(_agent_llm, _gen_llm, _crit_llm)

        model_log = {
            'model_planner':   f'{provider}::{agent_model}',
            'model_generator': f'{provider}::{gen_model}',
            'model_critic':    f'{provider}::{agent_model}',
        }

        tasks = _make_tasks(p_a, r_a, a_a, c_a)
        crew  = Crew(
            agents=[p_a, r_a, a_a, c_a],
            tasks=tasks,
            process=Process.sequential,
            verbose=CONFIG.get('verbose_eval_logging', True),
        )

        try:
            get_rate_limiter(provider).wait()
            crew_result = crew.kickoff()

            _preferred_models_by_task['planner']   = model_log['model_planner']
            _preferred_models_by_task['generator'] = model_log['model_generator']
            _preferred_models_by_task['critic']    = model_log['model_critic']

            latency = time.time() - start_time
            result  = _parse_crew_result(crew_result, question)
            result.update({'latency_sec': latency, **model_log})
            return result

        except Exception as e:
            last_error = e
            print(f'  [Crew] Error on {provider}:{agent_model}: {e}')
            if is_rate_limit_error(e) and attempt_idx < len(candidates) - 1:
                print('  [CrewFallback] Rate limit Ã¢â‚¬â€ trying next candidate')
                continue
            break

    # All attempts failed
    latency = time.time() - start_time
    err_str = str(last_error) if last_error else 'unknown error'
    return {
        'question':            question,
        'final_answer':        f'Error: {err_str}',
        'generated_answer':    '',
        'critic_decision':     'error',
        'critic_reason':       err_str,
        'repair_used':         False,
        'retrieved_chunks':    [],
        'retrieved_doc_names': [],
        'needs_decomposition': False,
        'sub_queries':         [],
        'latency_sec':         latency,
        'model_planner':       'error',
        'model_generator':     'error',
        'model_critic':        'error',
        'error':               err_str,
    }


print('Crew builder defined.')

In [ ]:
# Ã¢â€â‚¬Ã¢â€â‚¬ Runtime overrides (adjust without re-running full CONFIG cell) Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬

# Retrieval: cast a wider net before reranking.
# Original: bm25=8, dense=8, rerank=5  Ã¢â€ â€™  ~16-21 candidates (after RRF dedup).
# With adjacent expansion, pool grows by ~2x, so the cross-encoder sees ~40 candidates
# and has a real chance of finding the answer chunk even if it ranked 10-12 in BM25/dense.
CONFIG['bm25_top_k']   = 15
CONFIG['dense_top_k']  = 15
CONFIG['rerank_top_k'] = 7   # analyst/critic receive top-7 chunks

# inter_question_sleep_sec: each crew run makes ~8-12 internal LLM calls.
# At Gemini Free Tier 10 RPM, need ~60-70s between questions.
CONFIG['inter_question_sleep_sec'] = 65

print(f'bm25_top_k={CONFIG["bm25_top_k"]}  dense_top_k={CONFIG["dense_top_k"]}  rerank_top_k={CONFIG["rerank_top_k"]}')
print(f'inter_question_sleep_sec={CONFIG["inter_question_sleep_sec"]}s')

In [ ]:
# Ã¢â€â‚¬Ã¢â€â‚¬ Single Question Smoke Test Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
# test_q      = "What was NVIDIA's total revenue for fiscal year 2024?"
# test_golden = "NVIDIA's total revenue for fiscal year 2024 was $60.9 billion."

test_q      = "What were Apple's total net sales in fiscal year 2024?"
test_golden = "Apple's total net sales in FY2024 were $391.0 billion."

print(f'Question: {test_q}')
print('-' * 60)
result = build_and_run_crew(test_q, golden_answer=test_golden)

print()
print('=' * 60)
print(f'Final Answer:\n{result["final_answer"]}')
print(f'\nCritic  : {result["critic_decision"]} Ã¢â‚¬â€ {result.get("critic_reason", "")}')
print(f'Repair  : {result["repair_used"]}')
print(f'Latency : {result.get("latency_sec", 0):.1f}s')
print(f'Models  : planner={result.get("model_planner")}  gen={result.get("model_generator")}')
print(f'Docs    : {result["retrieved_doc_names"]}')
print(f'Decomp  : {result["needs_decomposition"]} | Sub-queries: {len(result["sub_queries"])}')

# Ã¢â€â‚¬Ã¢â€â‚¬ Optional: run judge separately on smoke test result Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
if test_golden and CONFIG.get('use_llm_judge', True):
    print('\nÃ¢â‚¬â€ LLM Judge (smoke test) Ã¢â‚¬â€')
    c_score, c_cov, c_reason, c_model = llm_judge_correctness(
        test_q, test_golden, result['final_answer']
    )
    print(f'Correctness : {c_score:.2f}  claims={c_cov:.2f}  ({c_model})')
    print(f'Reason      : {c_reason}')

    context_str = format_retrieved_context(
        result['retrieved_chunks'],
        max_chunks=CONFIG['judge_max_context_chunks'],
        max_chars=CONFIG['judge_max_context_chars'],
    )
    f_score, _, f_reason, f_model = llm_judge_faithfulness(
        test_q, context_str, result['final_answer']
    )
    print(f'Faithfulness: {f_score:.2f}  ({f_model})')
    print(f'Reason      : {f_reason}')

    tok_in, tok_out = _get_question_token_totals()
    print(f'\nTokens  : {tok_in} in / {tok_out} out  est=${_estimate_cost_usd(tok_in, tok_out):.5f}')

In [ ]:
# Ã¢â€â‚¬Ã¢â€â‚¬ Single Question Smoke Test Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
test_q      = "What was NVIDIA's total revenue for fiscal year 2024?"
test_golden = "NVIDIA's total revenue for fiscal year 2024 was $60.9 billion."

# test_q      = "What were Apple's total net sales in fiscal year 2024?"
# test_golden = "Apple's total net sales in FY2024 were $391.0 billion."

print(f'Question: {test_q}')
print('-' * 60)
result = build_and_run_crew(test_q, golden_answer=test_golden)

print()
print('=' * 60)
print(f'Final Answer:\n{result["final_answer"]}')
print(f'\nCritic  : {result["critic_decision"]} Ã¢â‚¬â€ {result.get("critic_reason", "")}')
print(f'Repair  : {result["repair_used"]}')
print(f'Latency : {result.get("latency_sec", 0):.1f}s')
print(f'Models  : planner={result.get("model_planner")}  gen={result.get("model_generator")}')
print(f'Docs    : {result["retrieved_doc_names"]}')
print(f'Decomp  : {result["needs_decomposition"]} | Sub-queries: {len(result["sub_queries"])}')

# Ã¢â€â‚¬Ã¢â€â‚¬ Optional: run judge separately on smoke test result Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
if test_golden and CONFIG.get('use_llm_judge', True):
    print('\nÃ¢â‚¬â€ LLM Judge (smoke test) Ã¢â‚¬â€')
    c_score, c_cov, c_reason, c_model = llm_judge_correctness(
        test_q, test_golden, result['final_answer']
    )
    print(f'Correctness : {c_score:.2f}  claims={c_cov:.2f}  ({c_model})')
    print(f'Reason      : {c_reason}')

    context_str = format_retrieved_context(
        result['retrieved_chunks'],
        max_chunks=CONFIG['judge_max_context_chunks'],
        max_chars=CONFIG['judge_max_context_chars'],
    )
    f_score, _, f_reason, f_model = llm_judge_faithfulness(
        test_q, context_str, result['final_answer']
    )
    print(f'Faithfulness: {f_score:.2f}  ({f_model})')
    print(f'Reason      : {f_reason}')

    tok_in, tok_out = _get_question_token_totals()
    print(f'\nTokens  : {tok_in} in / {tok_out} out  est=${_estimate_cost_usd(tok_in, tok_out):.5f}')

In [ ]:
# (duplicate smoke test Ã¢â‚¬â€ kept for quick re-runs)
# result = build_and_run_crew(test_q)
# print(result['final_answer'])

In [ ]:
# (duplicate smoke test Ã¢â‚¬â€ kept for quick re-runs)
# result = build_and_run_crew(test_q)
# print(result['final_answer'])

In [ ]:
eval_df = pd.read_csv('../sec_rag_team_share/evaluation/GenAI Eval QA.csv')

In [ ]:
eval_df.split.value_counts()

In [ ]:
# eval_df = eval_df[eval_df['split'].notnull()]

In [ ]:
target_split = CONFIG.get('eval_split', None)
if target_split:
    eval_df = eval_df[eval_df['split'] == target_split].copy()
    print(f"Filtered to split={target_split!r}: {len(eval_df)} questions")
else:
    eval_df = eval_df.copy()
    print(f"No split filter applied: {len(eval_df)} questions")

In [ ]:
eval_df.info()

In [ ]:
# Evaluation Loop (column schema matches langgraph_agentic_rag_sec_v3)
def run_evaluation(df: pd.DataFrame) -> pd.DataFrame:
    """
    Run the full CrewAI evaluation loop with resume support.

    Behavior:
      1. Loads existing results CSV (if present)
      2. Skips question_ids already present in results CSV
      3. Processes only pending questions
      4. Returns only newly processed rows for this run
    """
    all_results = []

    out_path = os.path.join(CONFIG['results_dir'], 'crewai_eval_results.csv')
    completed_ids = set()

    if os.path.exists(out_path):
        try:
            existing_df = pd.read_csv(out_path)
            if 'question_id' in existing_df.columns:
                completed_ids = set(
                    existing_df['question_id'].dropna().astype(str).str.strip().tolist()
                )
        except Exception as e:
            print(f"Warning: could not read existing results at {out_path}: {e}")

    work_df = df.copy()
    if 'question_id' not in work_df.columns:
        work_df['question_id'] = work_df.index.astype(str)

    work_df['question_id'] = work_df['question_id'].astype(str).str.strip()

    if completed_ids:
        before = len(work_df)
        work_df = work_df[~work_df['question_id'].isin(completed_ids)].copy()
        print(f"Resume mode: skipped {before - len(work_df)} already-processed questions")

    work_df = work_df.reset_index(drop=True)

    if work_df.empty:
        print("No pending questions to process.")
        return pd.DataFrame()

    print(f"Pending questions this run: {len(work_df)}")

    for i, row in tqdm(work_df.iterrows(), total=len(work_df), desc='CrewAI Eval'):
        question          = str(row.get('question', ''))
        reference_answer  = str(row.get('expected_answer', row.get('reference_answer', ''))).strip()
        question_id       = str(row.get('question_id', i)).strip()
        question_type     = row.get('question_type', 'unknown')
        company           = row.get('company',     '')
        ticker            = row.get('ticker',      '')
        form_type         = row.get('form_type',   '')
        filing_year       = row.get('filing_year', None)
        expected_decision = str(row.get('expected_decision', 'answer')).lower().strip()

        crew_result      = build_and_run_crew(question, golden_answer=None)
        final_answer     = crew_result.get('final_answer',     '')
        generated_answer = crew_result.get('generated_answer', '')
        retrieved_chunks = crew_result.get('retrieved_chunks', [])

        t_f1    = token_f1_score(final_answer, reference_answer) if reference_answer else None
        num_acc = numerical_accuracy_score(final_answer, reference_answer) if reference_answer else None
        predicted_decision = compute_decision_from_text(final_answer)
        decision_accuracy  = 1.0 if predicted_decision == expected_decision else 0.0

        context_str = format_retrieved_context(
            retrieved_chunks,
            max_chunks=CONFIG['judge_max_context_chunks'],
            max_chars=CONFIG['judge_max_context_chars'],
        )

        llm_correctness_score = llm_claims_covered = llm_correctness_reason = None
        model_judge_correctness = 'skipped'
        if CONFIG.get('use_llm_judge', True) and reference_answer:
            c_score, c_covered, c_reason, c_model = llm_judge_correctness(
                question, reference_answer, final_answer,
            )
            llm_correctness_score   = c_score
            llm_claims_covered      = c_covered
            llm_correctness_reason  = c_reason
            model_judge_correctness = c_model

        llm_faithfulness_score = llm_faithfulness_reason = None
        model_judge_faithfulness = 'skipped'
        if CONFIG.get('use_llm_judge', True):
            f_score, _, f_reason, f_model = llm_judge_faithfulness(
                question, context_str, final_answer,
            )
            llm_faithfulness_score   = f_score
            llm_faithfulness_reason  = f_reason
            model_judge_faithfulness = f_model

        crew_tok  = _question_tokens.get('crew',               {'input': 0, 'output': 0})
        corr_tok  = _question_tokens.get('judge_correctness',  {'input': 0, 'output': 0})
        faith_tok = _question_tokens.get('judge_faithfulness', {'input': 0, 'output': 0})
        total_in, total_out = _get_question_token_totals()
        est_cost            = _estimate_cost_usd(total_in, total_out)

        model_snapshot = get_task_model_snapshot(
            ['planner', 'generator', 'critic', 'judge_correctness', 'judge_faithfulness']
        )

        result_row = {
            'question_id':            question_id,
            'question':               question,
            'question_type':          question_type,
            'company':                company,
            'ticker':                 ticker,
            'form_type':              form_type,
            'filing_year':            filing_year,
            'reference_answer':       reference_answer,
            'expected_decision':      expected_decision,
            'final_answer':           final_answer,
            'generated_answer':       generated_answer,
            'pipeline':               'crewai_agentic_rag',
            'retrieved_doc_names':    ','.join(crew_result.get('retrieved_doc_names', [])),
            'needs_decomposition':    crew_result.get('needs_decomposition', False),
            'critic_decision':        crew_result.get('critic_decision', ''),
            'repair_used':            crew_result.get('repair_used', False),
            'latency_sec':            crew_result.get('latency_sec', None),
            'token_f1':               t_f1,
            'numerical_accuracy':     num_acc,
            'decision_accuracy':      decision_accuracy,
            'predicted_decision':     predicted_decision,
            'llm_correctness_score':  llm_correctness_score,
            'llm_claims_covered':     llm_claims_covered,
            'llm_correctness_reason': llm_correctness_reason,
            'llm_faithfulness_score': llm_faithfulness_score,
            'llm_faithfulness_reason': llm_faithfulness_reason,
            'model_planner':            crew_result.get('model_planner', model_snapshot.get('planner')),
            'model_generator':          crew_result.get('model_generator', model_snapshot.get('generator')),
            'model_critic':             crew_result.get('model_critic', model_snapshot.get('critic')),
            'model_judge_correctness':  model_judge_correctness,
            'model_judge_faithfulness': model_judge_faithfulness,
            'tokens_crew_input':         crew_tok['input'],
            'tokens_crew_output':        crew_tok['output'],
            'tokens_judge_corr_input':   corr_tok['input'],
            'tokens_judge_corr_output':  corr_tok['output'],
            'tokens_judge_faith_input':  faith_tok['input'],
            'tokens_judge_faith_output': faith_tok['output'],
            'tokens_total_input':        total_in,
            'tokens_total_output':       total_out,
            'est_cost_usd':              est_cost,
            'retrieved_chunks':          retrieved_chunks,
        }
        all_results.append(result_row)

    return pd.DataFrame(all_results)


print(f'Starting evaluation from input set of {len(eval_df)} questions...')
results_df = run_evaluation(eval_df)
print(f'\nThis run processed {len(results_df)} new questions.')

In [ ]:
results_df

In [ ]:
results_df.columns.tolist()

In [ ]:
results_df[['question', 'generated_answer', 'retrieved_chunks']]

In [ ]:
print(results_df.retrieved_chunks.iloc[2])

In [ ]:
results_df[['question', 'reference_answer', 'final_answer',  'generated_answer', 'expected_decision',
]]

In [ ]:
for idx, row in results_df.iterrows():
    print("Row:", idx)
    print("Question:", row['question'])
    print("Reference Answer:", row['reference_answer'])
    print("Final Answer:", row['final_answer'])
    print("Generated Answer:", row['generated_answer'])
    print("Expected Decision:", row['expected_decision'])
    print("-" * 50)

In [ ]:
import re
import pandas as pd

def clean_text(x):
    if not isinstance(x, str):
        return x

    # 1) turn literal escaped newlines/tabs into real ones
    x = x.replace("\\n", "\n").replace("\\t", "\t").replace("\\r", "\r")

    # 2) remove ASCII control chars except newline/tab
    x = re.sub(r'[\x00-\x08\x0b-\x1f\x7f-\x9f]', '', x)

    # 3) common mojibake repair for UTF-8 text decoded incorrectly
    try:
        repaired = x.encode("latin1").decode("utf-8")
        # only use repaired version if it looks better
        if repaired.count("Ã¯Â¿Â½") <= x.count("Ã¯Â¿Â½"):
            x = repaired
    except Exception:
        pass

    return x

results_df2 = results_df.copy()

for col in results_df2.columns:
    if pd.api.types.is_object_dtype(results_df2[col]):
        results_df2[col] = results_df2[col].apply(clean_text)

In [ ]:
results_df2.to_excel("crewai_results_clean.xlsx", index=False)

In [ ]:
results_df2

In [ ]:
# Ã¢â€â‚¬Ã¢â€â‚¬ Results & Metrics (matches langgraph_agentic_rag_sec_v3) Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
print('=' * 60)
print(' CREWAI AGENTIC RAG Ã¢â‚¬â€ EVALUATION SUMMARY')
print('=' * 60)

scored_c = results_df[results_df['llm_correctness_score'].notna()].copy()
scored_f = results_df[results_df['llm_faithfulness_score'].notna()].copy()

print(f'\nTotal questions       : {len(results_df)}')
print(f'Correctness judged    : {len(scored_c)}')
print(f'Faithfulness judged   : {len(scored_f)}')

# Ã¢â€â‚¬Ã¢â€â‚¬ Overall metrics Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
print('\nOverall metrics:')
for col, label in [
    ('llm_correctness_score',  'LLM Correctness  '),
    ('llm_claims_covered',     'Claims Covered   '),
    ('llm_faithfulness_score', 'LLM Faithfulness '),
    ('token_f1',               'Token F1         '),
    ('numerical_accuracy',     'Numerical Accuracy'),
    ('decision_accuracy',      'Decision Accuracy'),
]:
    vals = results_df[col].dropna()
    if len(vals):
        print(f'  {label}: {vals.mean():.4f}  (n={len(vals)})')

# Ã¢â€â‚¬Ã¢â€â‚¬ By question type Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
print('\nBy question_type:')
type_cols  = ['llm_correctness_score', 'llm_faithfulness_score', 'token_f1', 'decision_accuracy']
avail_cols = [c for c in type_cols if c in results_df.columns]
if avail_cols and 'question_type' in results_df.columns:
    by_type = results_df.groupby('question_type')[avail_cols].mean().round(3)
    print(by_type.to_string())

# Ã¢â€â‚¬Ã¢â€â‚¬ Critic decisions Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
print('\nCritic decision distribution:')
print(results_df['critic_decision'].value_counts().to_string())
print(f'\nRepair rate          : {results_df["repair_used"].mean():.1%}')
print(f'Decomposition rate   : {results_df["needs_decomposition"].mean():.1%}')

# Ã¢â€â‚¬Ã¢â€â‚¬ Decision accuracy Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
if 'decision_accuracy' in results_df.columns:
    print(f'\nDecision accuracy    : {results_df["decision_accuracy"].mean():.1%}')
    print('  Predicted distribution:')
    print(results_df['predicted_decision'].value_counts().to_string())

# Ã¢â€â‚¬Ã¢â€â‚¬ Model usage log Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
print('\nModel usage (most common per step):')
for col in ['model_planner', 'model_generator', 'model_critic',
            'model_judge_correctness', 'model_judge_faithfulness']:
    if col in results_df.columns and results_df[col].notna().any():
        top = results_df[col].value_counts().index[0]
        cnt = results_df[col].value_counts().iloc[0]
        print(f'  {col:28s}: {top}  ({cnt}/{len(results_df)})')

# Ã¢â€â‚¬Ã¢â€â‚¬ Latency Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
if 'latency_sec' in results_df.columns:
    lat = results_df['latency_sec'].dropna()
    if len(lat):
        print(f'\nCrew latency: mean={lat.mean():.1f}s  '
              f'median={lat.median():.1f}s  max={lat.max():.1f}s')

# Ã¢â€â‚¬Ã¢â€â‚¬ Token & cost summary Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
tok_cols = ['tokens_crew_input', 'tokens_crew_output',
            'tokens_judge_corr_input', 'tokens_judge_corr_output',
            'tokens_judge_faith_input', 'tokens_judge_faith_output',
            'tokens_total_input', 'tokens_total_output', 'est_cost_usd']
if all(c in results_df.columns for c in ['tokens_total_input', 'tokens_total_output', 'est_cost_usd']):
    print('\nToken & cost summary (across all questions):')
    total_in_all   = results_df['tokens_total_input'].sum()
    total_out_all  = results_df['tokens_total_output'].sum()
    total_cost_all = results_df['est_cost_usd'].sum()
    crew_in_all    = results_df['tokens_crew_input'].sum()
    crew_out_all   = results_df['tokens_crew_output'].sum()
    jc_in_all      = results_df['tokens_judge_corr_input'].sum()
    jc_out_all     = results_df['tokens_judge_corr_output'].sum()
    jf_in_all      = results_df['tokens_judge_faith_input'].sum()
    jf_out_all     = results_df['tokens_judge_faith_output'].sum()

    print(f'  Crew agents    : {crew_in_all:>8,} in  {crew_out_all:>7,} out')
    print(f'  Judge correctn : {jc_in_all:>8,} in  {jc_out_all:>7,} out')
    print(f'  Judge faithful : {jf_in_all:>8,} in  {jf_out_all:>7,} out')
    print(f'  TOTAL          : {total_in_all:>8,} in  {total_out_all:>7,} out')
    print(f'  Est. total cost: ${total_cost_all:.4f} USD')
    print(f'  Est. per-Q avg : ${results_df["est_cost_usd"].mean():.5f} USD')

    print(f'\nPer-question token stats:')
    for col in ['tokens_total_input', 'tokens_total_output', 'est_cost_usd']:
        vals = results_df[col].dropna()
        if len(vals):
            print(f'  {col:25s}: mean={vals.mean():.1f}  max={vals.max():.1f}')

# Ã¢â€â‚¬Ã¢â€â‚¬ Score distribution Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
if not scored_c.empty:
    print('\nCorrectness score distribution:')
    bins   = [0, 0.2, 0.4, 0.6, 0.8, 1.01]
    labels = ['0.0-0.2', '0.2-0.4', '0.4-0.6', '0.6-0.8', '0.8-1.0']
    scored_c['score_bin'] = pd.cut(
        scored_c['llm_correctness_score'], bins=bins, labels=labels, right=False
    )
    print(scored_c['score_bin'].value_counts().sort_index().to_string())

# Ã¢â€â‚¬Ã¢â€â‚¬ Save results Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
os.makedirs(CONFIG['results_dir'], exist_ok=True)
out_path = os.path.join(CONFIG['results_dir'], 'crewai_eval_results.csv')
new_export_df = results_df.drop(columns=['retrieved_chunks'], errors='ignore').copy()
if os.path.exists(out_path):
    existing_export_df = pd.read_csv(out_path)
    combined_df = pd.concat([existing_export_df, new_export_df], ignore_index=True)
else:
    combined_df = new_export_df
if 'question_id' in combined_df.columns:
    combined_df['question_id'] = combined_df['question_id'].astype(str).str.strip()
    combined_df = combined_df.drop_duplicates(subset=['question_id'], keep='last')
combined_df.to_csv(out_path, index=False)
print(f'\nResults saved -> {out_path}')
print(f'Total rows in saved CSV: {len(combined_df)}')
if 'question_id' in combined_df.columns and 'question_id' in eval_df.columns:
    source_ids = set(eval_df['question_id'].astype(str).str.strip())
    done_ids = set(combined_df['question_id'].astype(str).str.strip())
    print(f'Pending questions remaining from current eval_df: {len(source_ids - done_ids)}')

# Ã¢â€â‚¬Ã¢â€â‚¬ Preview Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬Ã¢â€â‚¬
preview_cols = [
    'question_id', 'question_type', 'ticker', 'critic_decision',
    'llm_correctness_score', 'llm_faithfulness_score', 'token_f1',
    'tokens_total_input', 'tokens_total_output', 'est_cost_usd',
    'model_planner', 'model_generator',
]
display_cols = [c for c in preview_cols if c in combined_df.columns]
combined_df[display_cols].head(10)

In [ ]:
# Batch-safe evaluation override: 4 batches x 20 questions
# Run this final cell instead of the older single-pass evaluation cell.

from pathlib import Path

BATCH_SIZE = 20
NUM_BATCHES = 4

def run_evaluation_batch(batch_df: pd.DataFrame, batch_index: int, total_batches: int) -> pd.DataFrame:
    all_results = []
    out_dir = Path(CONFIG['results_dir'])
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"crewai_eval_results_batch_{batch_index:02d}_of_{total_batches:02d}.csv"

    completed_ids = set()
    if out_path.exists():
        try:
            existing_df = pd.read_csv(out_path)
            if 'question_id' in existing_df.columns:
                completed_ids = set(existing_df['question_id'].dropna().astype(str).str.strip().tolist())
        except Exception as e:
            print(f"Warning: could not read existing batch file at {out_path}: {e}")

    work_df = batch_df.copy()
    if 'question_id' not in work_df.columns:
        work_df['question_id'] = work_df.index.astype(str)
    work_df['question_id'] = work_df['question_id'].astype(str).str.strip()

    if completed_ids:
        before = len(work_df)
        work_df = work_df[~work_df['question_id'].isin(completed_ids)].copy()
        print(f"Batch {batch_index}/{total_batches}: resumed and skipped {before - len(work_df)} already-processed questions")

    work_df = work_df.reset_index(drop=True)
    if work_df.empty:
        print(f"Batch {batch_index}/{total_batches}: no pending questions.")
        try:
            return pd.read_csv(out_path) if out_path.exists() else pd.DataFrame()
        except Exception:
            return pd.DataFrame()

    print(f"Batch {batch_index}/{total_batches}: pending questions = {len(work_df)}")

    for i, row in tqdm(work_df.iterrows(), total=len(work_df), desc=f'CrewAI Eval B{batch_index}/{total_batches}'):
        question = str(row.get('question', ''))
        reference_answer = str(row.get('expected_answer', row.get('reference_answer', ''))).strip()
        question_id = str(row.get('question_id', i)).strip()
        question_type = row.get('question_type', 'unknown')
        company = row.get('company', '')
        ticker = row.get('ticker', '')
        form_type = row.get('form_type', '')
        filing_year = row.get('filing_year', None)
        expected_decision = str(row.get('expected_decision', 'answer')).lower().strip()

        crew_result = build_and_run_crew(question, golden_answer=None)
        final_answer = crew_result.get('final_answer', '')
        generated_answer = crew_result.get('generated_answer', '')
        retrieved_chunks = crew_result.get('retrieved_chunks', [])

        t_f1 = token_f1_score(final_answer, reference_answer) if reference_answer else None
        num_acc = numerical_accuracy_score(final_answer, reference_answer) if reference_answer else None
        predicted_decision = compute_decision_from_text(final_answer)
        decision_accuracy = 1.0 if predicted_decision == expected_decision else 0.0

        context_str = format_retrieved_context(
            retrieved_chunks,
            max_chunks=CONFIG['judge_max_context_chunks'],
            max_chars=CONFIG['judge_max_context_chars'],
        )

        llm_correctness_score = llm_claims_covered = llm_correctness_reason = None
        model_judge_correctness = 'skipped'
        if CONFIG.get('use_llm_judge', True) and reference_answer:
            c_score, c_covered, c_reason, c_model = llm_judge_correctness(
                question, reference_answer, final_answer,
            )
            llm_correctness_score = c_score
            llm_claims_covered = c_covered
            llm_correctness_reason = c_reason
            model_judge_correctness = c_model

        llm_faithfulness_score = llm_faithfulness_reason = None
        model_judge_faithfulness = 'skipped'
        if CONFIG.get('use_llm_judge', True):
            f_score, _, f_reason, f_model = llm_judge_faithfulness(
                question, context_str, final_answer,
            )
            llm_faithfulness_score = f_score
            llm_faithfulness_reason = f_reason
            model_judge_faithfulness = f_model

        crew_tok = _question_tokens.get('crew', {'input': 0, 'output': 0})
        corr_tok = _question_tokens.get('judge_correctness', {'input': 0, 'output': 0})
        faith_tok = _question_tokens.get('judge_faithfulness', {'input': 0, 'output': 0})
        total_in, total_out = _get_question_token_totals()
        est_cost = _estimate_cost_usd(total_in, total_out)

        model_snapshot = get_task_model_snapshot(
            ['planner', 'generator', 'critic', 'judge_correctness', 'judge_faithfulness']
        )

        # Intentionally do not store retrieved_chunks in output rows to avoid memory bloat.
        result_row = {
            'question_id': question_id,
            'question': question,
            'question_type': question_type,
            'company': company,
            'ticker': ticker,
            'form_type': form_type,
            'filing_year': filing_year,
            'reference_answer': reference_answer,
            'expected_decision': expected_decision,
            'final_answer': final_answer,
            'generated_answer': generated_answer,
            'pipeline': 'crewai_agentic_rag',
            'retrieved_doc_names': ','.join(crew_result.get('retrieved_doc_names', [])),
            'needs_decomposition': crew_result.get('needs_decomposition', False),
            'critic_decision': crew_result.get('critic_decision', ''),
            'repair_used': crew_result.get('repair_used', False),
            'latency_sec': crew_result.get('latency_sec', None),
            'token_f1': t_f1,
            'numerical_accuracy': num_acc,
            'decision_accuracy': decision_accuracy,
            'predicted_decision': predicted_decision,
            'llm_correctness_score': llm_correctness_score,
            'llm_claims_covered': llm_claims_covered,
            'llm_correctness_reason': llm_correctness_reason,
            'llm_faithfulness_score': llm_faithfulness_score,
            'llm_faithfulness_reason': llm_faithfulness_reason,
            'model_planner': crew_result.get('model_planner', model_snapshot.get('planner')),
            'model_generator': crew_result.get('model_generator', model_snapshot.get('generator')),
            'model_critic': crew_result.get('model_critic', model_snapshot.get('critic')),
            'model_judge_correctness': model_judge_correctness,
            'model_judge_faithfulness': model_judge_faithfulness,
            'tokens_crew_input': crew_tok['input'],
            'tokens_crew_output': crew_tok['output'],
            'tokens_judge_corr_input': corr_tok['input'],
            'tokens_judge_corr_output': corr_tok['output'],
            'tokens_judge_faith_input': faith_tok['input'],
            'tokens_judge_faith_output': faith_tok['output'],
            'tokens_total_input': total_in,
            'tokens_total_output': total_out,
            'est_cost_usd': est_cost,
            'batch_index': batch_index,
        }
        all_results.append(result_row)

    new_export_df = pd.DataFrame(all_results)
    if out_path.exists():
        try:
            existing_export_df = pd.read_csv(out_path)
            combined_df = pd.concat([existing_export_df, new_export_df], ignore_index=True)
        except Exception:
            combined_df = new_export_df.copy()
    else:
        combined_df = new_export_df.copy()

    if 'question_id' in combined_df.columns:
        combined_df['question_id'] = combined_df['question_id'].astype(str).str.strip()
        combined_df = combined_df.drop_duplicates(subset=['question_id'], keep='last')

    combined_df.to_csv(out_path, index=False)
    print(f"Batch {batch_index}/{total_batches} saved -> {out_path} | rows={len(combined_df)}")
    return combined_df

def run_evaluation_in_batches(df: pd.DataFrame, batch_size: int = 20, num_batches: int = 4) -> pd.DataFrame:
    target_n = batch_size * num_batches
    work_df = df.copy().reset_index(drop=True).head(target_n)

    if work_df.empty:
        print('No rows found in eval_df.')
        return pd.DataFrame()

    print(
        f"Starting batched evaluation: total_rows={len(work_df)}, "
        f"batch_size={batch_size}, batches={num_batches}"
    )

    all_batch_results = []
    for b in range(num_batches):
        start = b * batch_size
        end = min(start + batch_size, len(work_df))
        if start >= len(work_df):
            break
        batch_df = work_df.iloc[start:end].copy()
        print(f"\n--- Running batch {b + 1}/{num_batches} on rows [{start}:{end}] (n={len(batch_df)}) ---")
        batch_results = run_evaluation_batch(batch_df, b + 1, num_batches)
        if not batch_results.empty:
            all_batch_results.append(batch_results)

    if not all_batch_results:
        return pd.DataFrame()

    merged = pd.concat(all_batch_results, ignore_index=True)
    if 'question_id' in merged.columns:
        merged['question_id'] = merged['question_id'].astype(str).str.strip()
        merged = merged.drop_duplicates(subset=['question_id'], keep='last').reset_index(drop=True)

    return merged

results_df = run_evaluation_in_batches(eval_df, batch_size=BATCH_SIZE, num_batches=NUM_BATCHES)
saved_batch_paths = [
    str(Path(CONFIG['results_dir']) / f"crewai_eval_results_batch_{i:02d}_of_{NUM_BATCHES:02d}.csv")
    for i in range(1, NUM_BATCHES + 1)
]

print(f"\nBatched run complete. Total rows currently loaded in results_df: {len(results_df)}")
print('Batch files:')
for p in saved_batch_paths:
    print(f"  - {p}")
